# Pseudobulk Preparation From The Annotated Harmony Workflow

In this notebook, I prepare donor-level pseudobulk inputs from the current Harmony annotation workflow. I use:

- `adata_hm_annotated.h5ad` for the final Harmony-space annotations
- `adata_full_for_annotation.h5ad` for the full-gene raw-count matrix

This keeps the roles of the two objects clear: `adata_hm` stores the clustering and manual annotations, whereas the full-gene object provides the count matrix needed for pseudobulk aggregation and downstream DE analysis.


In [1]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc


In [2]:
import sys
sys.path.append(str(Path("../src").resolve()))
from c9_snrnaseq.io_utils import PROJECT_ROOT


## 1. Load The Annotated Harmony Object And The Full-Gene Count Object

I load the final annotated Harmony object together with the full-gene object prepared in notebook 09. The full-gene object is the pseudobulk count source; the Harmony object is the annotation source of truth.


In [3]:
HM_ANNOT_PATH = PROJECT_ROOT / "data/processed/merged/adata_hm_annotated.h5ad"
FULL_ANNOT_PATH = PROJECT_ROOT / "data/processed/merged/adata_full_for_annotation.h5ad"

adata_hm = sc.read_h5ad(HM_ANNOT_PATH)
adata_full = sc.read_h5ad(FULL_ANNOT_PATH)

print("adata_hm:", adata_hm.shape)
print("adata_full:", adata_full.shape)
print("adata_hm has cell_class_major_harmony:", "cell_class_major_harmony" in adata_hm.obs.columns)
print("adata_hm has cell_subtype:", "cell_subtype" in adata_hm.obs.columns)
print("adata_full has counts layer:", "counts" in adata_full.layers)
print("adata_full has leiden_harmony:", "leiden_harmony" in adata_full.obs.columns)


adata_hm: (88883, 2000)
adata_full: (88883, 61552)
adata_hm has cell_class_major_harmony: True
adata_hm has cell_subtype: True
adata_full has counts layer: True
adata_full has leiden_harmony: True


In [4]:
# I require the full-gene object and the annotated Harmony object to refer to the
# same cells in the same order. If they differ, I stop here and rebuild the
# full-gene object from notebook 09 so that pseudobulk counts remain trustworthy.

if not adata_hm.obs_names.equals(adata_full.obs_names):
    raise ValueError(
        "adata_hm_annotated.h5ad and adata_full_for_annotation.h5ad are not aligned. "
        "Please rerun notebook 09 with REBUILD_FULL_ANNOTATION_OBJECT=True and then rerun this notebook."
    )

required_hm_cols = ["leiden_harmony", "cell_class_major_harmony", "cell_subtype", "sample", "condition"]
missing_hm_cols = [col for col in required_hm_cols if col not in adata_hm.obs.columns]
if missing_hm_cols:
    raise KeyError(f"adata_hm is missing required annotation columns: {missing_hm_cols}")

if "counts" not in adata_full.layers:
    if "raw_counts" in adata_full.layers:
        adata_full.layers["counts"] = adata_full.layers["raw_counts"].copy()
    else:
        adata_full.layers["counts"] = adata_full.X.copy()

adata_merged_pb = adata_full.copy()

# I copy the final annotation columns from adata_hm onto the full-gene object.
# The alignment check above lets me assign by position safely.
for col in ["leiden_harmony", "cell_class_major_harmony", "cell_subtype", "sample", "condition"]:
    adata_merged_pb.obs[col] = adata_hm.obs[col].values

adata_merged_pb.obs["barcode"] = adata_merged_pb.obs_names.astype(str)
adata_merged_pb.obs["merge_key"] = adata_merged_pb.obs_names.astype(str)

print("Cells in pseudobulk source object:", adata_merged_pb.n_obs)
print(adata_merged_pb.obs["cell_class_major_harmony"].value_counts(dropna=False))
print(adata_merged_pb.obs["cell_subtype"].value_counts(dropna=False).head(20))


Cells in pseudobulk source object: 88883
cell_class_major_harmony
Glia          65529
Excitatory    16001
Inhibitory     6717
Vascular        636
Name: count, dtype: int64
cell_subtype
Glia.Oligo                 50041
Ex.L2_L3.CUX2_RASGRF2       6768
Glia.Astro.GFAP.neg         5246
Glia.OPC                    5143
Glia.Micro                  3327
In.5HT3aR.DISC1_RELN        2464
Ex.L4_L6.RORB_LRRK1         1908
In.PV.PVALB_MYBPC1          1884
Ex.L5_L6.THEMIS_TMEM233     1818
Ex.L4_L5.RORB_POU3F2        1757
Glia.Astro.GFAP.pos         1588
In.SOM.SST_ADAMTS19         1405
Ex.L3_L5.CUX2_RORB          1065
In.Rosehip.LAMP5_PMEPA1      964
Ex.L6.TLE4_MEGF11            816
Ex.L6.TLE4_CCBE1             703
Ex.L5.PCP4_NXPH2             529
Ex.L5_L6.THEMIS_NR4A2        431
Vasc.Endo.Venous             367
Vasc.Mural.Pericyte          269
Name: count, dtype: int64


## 2. Define Small Helper Functions

I keep the pseudobulk logic explicit and donor-based so the outputs are easy to inspect and reuse across downstream notebooks.


In [5]:
def slugify_label(label):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(label)).strip("_")


def pseudobulk_by_sample(
    adata,
    group_value,
    groupby_col="cell_class_major_harmony",
    min_cells_per_sample=20,
    layer="counts",
):
    sub = adata[adata.obs[groupby_col].astype(str) == str(group_value)].copy()
    if sub.n_obs == 0:
        raise ValueError(f"No cells found for {groupby_col}={group_value!r}.")

    cell_counts = sub.obs["sample"].astype(str).value_counts()
    keep_samples = cell_counts[cell_counts >= min_cells_per_sample].index.tolist()
    if not keep_samples:
        raise ValueError(
            f"No samples retained for {groupby_col}={group_value!r} "
            f"with min_cells_per_sample={min_cells_per_sample}."
        )

    sub = sub[sub.obs["sample"].astype(str).isin(keep_samples)].copy()
    sample_order = sorted(sub.obs["sample"].astype(str).unique())

    pb_counts = []
    pb_meta = []

    for samp in sample_order:
        ss = sub[sub.obs["sample"].astype(str) == samp]
        matrix = ss.layers[layer] if layer in ss.layers else ss.X
        vec = np.asarray(matrix.sum(axis=0)).ravel()

        pb_counts.append(vec)
        pb_meta.append({
            "sample": samp,
            "condition": ss.obs["condition"].astype(str).iloc[0],
            "group": str(group_value),
            "groupby_col": groupby_col,
            "n_cells": ss.n_obs,
        })

    counts_df = pd.DataFrame(pb_counts, index=sample_order, columns=sub.var_names)
    meta_df = pd.DataFrame(pb_meta).set_index("sample")
    return counts_df, meta_df


def save_pseudobulk_panel(
    adata,
    groups,
    out_dir,
    groupby_col,
    min_cells_per_sample=20,
    layer="counts",
):
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = []
    skipped = []

    for group_value in groups:
        try:
            counts_df, meta_df = pseudobulk_by_sample(
                adata,
                group_value=group_value,
                groupby_col=groupby_col,
                min_cells_per_sample=min_cells_per_sample,
                layer=layer,
            )
        except ValueError as exc:
            skipped.append((group_value, str(exc)))
            print(f"Skipped {group_value}: {exc}")
            continue

        stem = slugify_label(group_value)
        counts_path = out_dir / f"{stem}_counts.csv"
        meta_path = out_dir / f"{stem}_meta.csv"

        counts_df.to_csv(counts_path)
        meta_df.to_csv(meta_path)

        saved.append((group_value, counts_df.shape, meta_df.shape))
        print(f"Saved pseudobulk for {group_value}")
        print(f"  counts: {counts_path}")
        print(f"  meta:   {meta_path}")

    return saved, skipped


## 3. Export Broad-Class Pseudobulk Matrices

I first create donor-level pseudobulk matrices for the four broad classes used in notebook 09. These are the highest-level summaries of the current annotation workflow.


In [6]:
RESULTS_DIR = PROJECT_ROOT / "results"
PSEUDOBULK_DIR = RESULTS_DIR / "pseudobulk"
PSEUDOBULK_BROAD_DIR = PSEUDOBULK_DIR / "broad"

broad_main_classes = [
    label for label in ["Excitatory", "Inhibitory", "Glia", "Vascular"]
    if label in set(adata_merged_pb.obs["cell_class_major_harmony"].astype(str))
]

broad_saved, broad_skipped = save_pseudobulk_panel(
    adata_merged_pb,
    groups=broad_main_classes,
    out_dir=PSEUDOBULK_BROAD_DIR,
    groupby_col="cell_class_major_harmony",
    min_cells_per_sample=20,
    layer="counts",
)

print("Broad classes saved:", broad_saved)
print("Broad classes skipped:", broad_skipped)


Saved pseudobulk for Excitatory
  counts: /Users/wangj/Documents/Computational_Biology/Projects/C9_Multiomics/results/pseudobulk/broad/Excitatory_counts.csv
  meta:   /Users/wangj/Documents/Computational_Biology/Projects/C9_Multiomics/results/pseudobulk/broad/Excitatory_meta.csv
Saved pseudobulk for Inhibitory
  counts: /Users/wangj/Documents/Computational_Biology/Projects/C9_Multiomics/results/pseudobulk/broad/Inhibitory_counts.csv
  meta:   /Users/wangj/Documents/Computational_Biology/Projects/C9_Multiomics/results/pseudobulk/broad/Inhibitory_meta.csv
Saved pseudobulk for Glia
  counts: /Users/wangj/Documents/Computational_Biology/Projects/C9_Multiomics/results/pseudobulk/broad/Glia_counts.csv
  meta:   /Users/wangj/Documents/Computational_Biology/Projects/C9_Multiomics/results/pseudobulk/broad/Glia_meta.csv
Saved pseudobulk for Vascular
  counts: /Users/wangj/Documents/Computational_Biology/Projects/C9_Multiomics/results/pseudobulk/broad/Vascular_counts.csv
  meta:   /Users/wangj/Do

## 4. Save The Broad-Class Pseudobulk Source Object

I save the full-gene object with the current broad Harmony labels so downstream DE notebooks can start from a stable broad-class pseudobulk-ready checkpoint.


In [7]:
pb_ckpt = PROJECT_ROOT / "data/processed/merged/adata_merged_pb.h5ad"
pb_ckpt.parent.mkdir(parents=True, exist_ok=True)

adata_merged_pb.write_h5ad(pb_ckpt)

print(f"Saved broad-class pseudobulk source object to: {pb_ckpt}")
print(adata_merged_pb.shape)
print("Has cell_class_major_harmony:", "cell_class_major_harmony" in adata_merged_pb.obs.columns)
print("Has counts layer:", "counts" in adata_merged_pb.layers)
print("Broad classes:", sorted(adata_merged_pb.obs["cell_class_major_harmony"].dropna().astype(str).unique()))


Saved broad-class pseudobulk source object to: /Users/wangj/Documents/Computational_Biology/Projects/C9_Multiomics/data/processed/merged/adata_merged_pb.h5ad
(88883, 61552)
Has cell_class_major_harmony: True
Has counts layer: True
Broad classes: ['Excitatory', 'Glia', 'Inhibitory', 'Vascular']
